# 리포트 83 — 나딧에서 레벨을 −130.78 dB 에서 −64.23 dB 로 올리는 스위치는 회절 하나다

> ### 한 일
> **앙각 스윕이 두 엔진에 넘긴 물리 스위치를 소스에서 읽어 표로 적고, PathSolver 의 네 스위치를 하나씩만 켜서 나딧 레벨의 상승을 귀속했다.**

### 결과
1. 스윕이 돈 기준 판은 레벨 -130.78 dB [^1] · 경로 중앙값 11 개 [^2] 다.
2. 회절만 켠 판은 -64.23 dB [^3] 로 올라가고, 네 스위치를 전부 켠 판도 -64.23 dB [^4] 로 같은 자리에 선다.
3. 굴절만 켠 판 -132.48 dB [^5] · 다중반사만 켠 판 -130.76 dB [^6] · 모서리회절만 켠 판 -130.78 dB [^7] 는 기준 자리에 머문다.
4. 기준 실행의 투과 축에서는 우리 팔이 켠 물리가 하나 더 많다 — 우리 팔은 `penetrate=True`, 그 PathSolver 팔은 `refraction=False` 다.
5. 두 팔 다 거리 10 m [^8] 의 실제 기하로 위상을 준다 — 평면파 근사는 두 팔의 이번 설정 밖에 있다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 우리 팔의 설정 | `benchmark/elevation_sweep_md.py:134`~`135` 의 `sbr_field(…)` 호출 인자를 그대로 읽었다 — `penetrate` 는 기본값 True(`src/rcs_sbr.py:1009`), `ptd` 는 False, `range_m` 은 10 m 고정(`benchmark/elevation_sweep_md.py:74`) |
| PathSolver 의 설정 | `benchmark/elevation_sweep_md.py:176`~`183` 의 호출 인자. 스위치 넷은 `--physics` 플래그 하나에 묶여 있어 팔이 두 갈래다 — 없이 돌면 `max_depth=1`·굴절·회절·모서리회절 모두 끔, 주면 `max_depth=3` 에 셋 다 켬 |
| 스톡 기본값과의 거리 | Sionna 2.0.1 의 `PathSolver.__call__` 기본값은 `max_depth=3` · `refraction=True` · `diffuse_reflection=False` 다(`sionna/rt/path_solvers/path_solver.py:146`·`152`~`155`) — 이 스윕의 기준 팔은 그 기본값에서 세 칸을 옮긴 설정이다 |
| 스위치 귀속 | `benchmark/diag_physics_paths.py:35`~`42` 가 자세·광선 예산·거리를 고정하고 스위치 하나씩만 바꿔 여섯 판을 돌린 원장 |
| 경로 수를 읽는 법 | 같은 엔진이 PEC 구와 챔버 판에서 낸 경로 수 — `outputs/rt_no_rcs_verify.json` |

### 재현

```bash
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/diag_physics_paths.py -90 20
```

| | |
|---|---|
| 출력 | `outputs/diag_physics_paths_el-90.json` |
| 소요 | 여섯 판 × 자세 20 개 (GPU 1 장) |
| 비고 | 자세 수만 줄인 진단 판이다 — 광선 예산·거리·기체는 앙각 스윕 본판과 같다 |

---

## 두 엔진에 실제로 넘어간 스위치

엔진이 «할 수 있는 것»과 «이번에 한 것»은 다른 물음이다. 아래 표는 그 둘을 갈라 적는다 — 왼쪽 두 칸이 우리 팔(SBR + 물리광학), 오른쪽 두 칸이 스톡 PathSolver 다.

낱말부터 푼다. **가림**은 앞면이 뒷면을 막는 것, **투과**는 플라스틱 셸을 지나 안쪽 금속(배터리·PCB)까지 세는 것, **쐐기 회절**은 실루엣 모서리에서 빛이 꺾여 도는 것, **다중반사**는 표적 안에서 두 번 이상 튀는 것이다.

## 스위치 대조표

| 물리 | 우리 팔이 담을 수 있나 | 이번에 켰나 | PathSolver 가 담을 수 있나 | 이번에 켰나 |
|---|---|---|---|---|
| 가림 | 담는다 — 첫 충돌만 채택(`src/rcs_sbr.py:1068`) | 켬 | 담는다 — 광선추적 그 자체 | 켬 |
| 산란 표면적분(PO) | 담는다 — 조명면 적분(`src/rcs_sbr.py:1101`) | 켬 | 스톡에는 그 단계가 빠져 있다 | — |
| 투과(셸 → 내부 금속) | 담는다 — τ=1−\|Γ\|² 로 두 번째 패스(`src/rcs_sbr.py:1105`) | 켬 | 담는다 — `refraction` | 끔 |
| 쐐기 회절 | 담는다 — PTD 프린지(`src/rcs_sbr.py:1112`) | 끔 | 담는다 — `diffraction` | 끔 |
| 자유 모서리 회절 | PTD 프린지 항이 그 자리를 맡는다 | 끔 | 담는다 — `edge_diffraction` | 끔 |
| 다중반사 | 담는다 — `rcs_sbr(max_bounce≥2)`(`src/rcs_sbr.py:1302`) | 끔 — 스윕은 1차 히트 커널을 부른다 | 담는다 — `max_depth` | 끔 — `max_depth=1` |
| 확산 반사 | PO 적분이 그 자리를 맡는다 | — | 담는다 — `diffuse_reflection` | 켬 |
| 구면파 조명 | 담는다 — `range_m`(`src/rcs_sbr.py:1093`) | 켬 | 송수신점을 실제 자리에 놓는다 | 켬 |

## 이 표가 서는 범위 — 어느 팔을 말하는가

위 표의 «이번에 켰나» 칸은 **기준 실행**(`--physics` 없이 돈 `sionna` 계열 네 팔)의 상태다. 같은 스크립트가 `--physics` 로 돌린 팔이 둘 더 있고, 그 팔은 같은 줄에서 `max_depth=3` 에 굴절·회절·모서리회절을 전부 켠다. 원장에도 그 팔의 행이 있다 — `engine="sionna_phys"` 7 행이고 그중 7 행이 `n_missing = 0` 인 완결 행이다[^9].

그래서 축마다 성립 범위가 다르다.

- **다중반사** — 기준 실행에서는 두 팔 다 끔이라 사과-대-사과다. `--physics` 판에서는 PathSolver 만 깊이 3 이 되고, 우리 팔은 스윕이 부르는 1 차 히트 커널에 다중반사 루프 자체가 없다(`src/rcs_sbr.py:1068`·투과 패스 `:1105`; 루프가 있는 `rcs_sbr(max_bounce≥2)` 는 스윕이 부르지 않는다).
- **투과** — 기준 실행에서는 우리 팔만 셸 안을 본다. 그 τ 는 수직입사 |Γ| 로 만든 값이고 굴절각·유전체 내 위상지연·두께는 들어가지 않는다.
- **조명** — 두 팔 다 실제 기하로 위상을 준다. 우리 커널의 평면파 갈래는 `range_m=None` 일 때만 켜지는 별개의 길이다.

⚠ 10 m 는 이 기체의 원거리장 경계 안쪽이다 — 두 팔 다 근거리장 판을 계산한 것이고, 원거리장 평면파 값과 나란히 놓는 비교는 이 절의 범위 밖이다.

## 스위치를 하나씩 켠 여섯 판

| 설정 | 경로 중앙값 | 레벨 [dB] | 자세당 초 |
|---|---|---|---|
| 기준(지금까지의 실행) | 11 | -130.78 | 0.263 |
| 굴절만 켬 | 2 | -132.48 | 0.274 |
| 회절만 켬 | 14 | -64.23 | 0.423 |
| 모서리회절만 켬 | 11 | -130.78 | 0.250 |
| 다중반사만 (depth 3) | 12 | -130.76 | 0.302 |
| 전부 켬 (--physics) | 6 | -64.23 | 0.310 |

출처 [^10]

![switch axis](../outputs/figures/report17_switch_axis.png)

**그림 1.** 네 스위치 중 어느 것이 나딧 레벨을 올리나?

회절만 켠 판 -64.23 dB [^3] 은 기준 판 -130.78 dB [^1] 보다 **66.55 dB** 위에 선다(두 칸의 차). 나머지 세 스위치는 기준 자리에 머물고, 네 스위치를 전부 켠 판의 레벨은 회절만 켠 판과 같다 — `--physics` 가 올린 것은 회절 하나다.

## 모서리 회절만 켠 판이 기준과 같은 이유

모서리회절만 켠 판은 레벨 -130.78 dB [^7] · 경로 중앙값 11 개 [^11] 로 기준 판과 같은 자리에 선다.

이유는 엔진 소스에 적혀 있다. `edge_diffraction` 은 `sionna/rt/path_solvers/sb_candidate_generator.py:600` 의 `if diffraction_enabled:` 블록 **안에서만** 읽힌다. 즉 `diffraction=False` 인 채로 `edge_diffraction=True` 를 주면 그 인자는 자유 모서리를 여는 대신 그냥 지나간다 — 이 행은 스위치의 크기가 아니라 **설정의 항등식**이고, 세 앙각 전부에서 네 값이 똑같이 겹치는 것이 그 표시다.

⭐그러니 `--physics` 의 네 스위치 중 실제로 레벨을 만든 것은 회절 하나이고, 모서리 회절의 크기는 `diffraction=True` 를 고정한 채 그 스위치만 뒤집는 판에서 잰다. 그 판은 다음 단계 표에 있다.

스톡 Sionna 2.0.1 이 여는 회절은 **1 차 한 번까지**다 — 소스가 «Only first order diffraction is supported» 라 적고(`sb_candidate_generator.py:394`), 회절을 뽑은 경로에서는 이후 회절과 확산반사를 끈다(같은 파일 `:405`~`:407`).

## 경로 수는 스위치의 지표이고, 세기는 레벨 칸이 잰다

이 절의 표에 경로 수를 함께 적은 것은 스위치가 무엇을 열었는지 보이기 위해서다. 같은 엔진이 반지름 0.3 m [^12] 짜리 금속 구에 광선 1,000,000 발 [^13] 을 쏘아 얻은 표적 경로는 0 개 [^14] 이고, 챔버 판의 표적 근처 경로 12 개 [^15] 중 진짜 에코는 1 개 [^16] 다.

경로 수와 레벨이 갈라지는 자리도 이 표 안에 있다 — 전부 켠 판은 경로 중앙값 6 개 [^17] 로 회절만 켠 판의 14 개 [^18] 보다 적은데 레벨은 같다.

스톡 솔버의 경로 진폭이 표적 면적을 어떻게 따라가는지는 [편 05 «면적을 키운 스윕»](05_size-sweep.ipynb) 이 이미 재 놓았다. 여기서는 그 결론을 **경로 수를 읽는 주의**로만 쓴다.

## 이 절이 고정한 것

① 두 팔의 스위치 상태는 소스 줄 번호로 고정됐다. ② `--physics` 의 상승은 회절 하나에 귀속된다. ③ 경로 수는 스위치의 지표이고, 세기는 레벨 칸이 잰다.

회절이 올린 그 상승분이 표적에서 온 것인지 정적 성분에서 온 것인지는 다음 절이 분모를 갈라 답한다 — [편 84 «분모가 커진 결과»](84_physics-denominator.ipynb).

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 우리 팔의 모서리 회절을 켜고(`--ptd`) 같은 나딧 판을 다시 돌린다 | 두 팔이 모서리 회절 축에서 같은 설정이 되어 회절 상승을 팔 사이에서 비교할 수 있다 | `benchmark/elevation_sweep_md.py --ptd` |
| `diffraction=True` 를 고정한 채 `edge_diffraction` 만 껐다 켠 판을 돌린다 | 모서리 회절 스위치 자체의 크기가 처음으로 수치로 나온다 | `benchmark/diag_physics_paths.py` 케이스 추가 · **새 계산이 필요하다** |
| 회절을 켠 판의 시계열을 대역 잣대로 읽는다 | 회절이 올린 전력이 날개끝 상한 위로 새는 몫인지가 수치로 갈린다 | [편 85 «상한 위 에너지 몫»](85_physics-above-limit.ipynb) |
| 굴절만 켠 판을 앙각 일곱 점으로 넓힌다 | 투과 축의 두 팔 차이가 앙각마다 확정된다 | `benchmark/diag_physics_paths.py <el>` |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 18개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/diag_physics_paths_el-90.json` | `cases.기준(지금까지의 실행).level_db` | -130.8 |
| [^2] | `outputs/diag_physics_paths_el-90.json` | `cases.기준(지금까지의 실행).npaths_median` | 11 |
| [^3] | `outputs/diag_physics_paths_el-90.json` | `cases.회절만 켬.level_db` | -64.23 |
| [^4] | `outputs/diag_physics_paths_el-90.json` | `cases.전부 켬 (--physics).level_db` | -64.23 |
| [^5] | `outputs/diag_physics_paths_el-90.json` | `cases.굴절만 켬.level_db` | -132.5 |
| [^6] | `outputs/diag_physics_paths_el-90.json` | `cases.다중반사만 (depth 3).level_db` | -130.8 |
| [^7] | `outputs/diag_physics_paths_el-90.json` | `cases.모서리회절만 켬.level_db` | -130.8 |
| [^8] | `outputs/diag_physics_paths_el-90.json` | `_meta.range_m` | 10 |
| [^9] | `outputs/elevation_sweep_md.json` | `rows → engine=sionna_phys 행 수와 그중 n_missing=0 인 행 수` | (47행 표) (파생) |
| [^10] | `outputs/diag_physics_paths_el-90.json` | `cases` | (6항목 묶음) |
| [^11] | `outputs/diag_physics_paths_el-90.json` | `cases.모서리회절만 켬.npaths_median` | 11 |
| [^12] | `outputs/rt_no_rcs_verify.json` | `C_pec_sphere[0].r` | 0.3 |
| [^13] | `outputs/rt_no_rcs_verify.json` | `C_pec_sphere[0].spp` | 1000000 |
| [^14] | `outputs/rt_no_rcs_verify.json` | `C_pec_sphere[0].n_paths` | 0 |
| [^15] | `outputs/rt_no_rcs_verify.json` | `D_chamber_paths.n_near` | 12 |
| [^16] | `outputs/rt_no_rcs_verify.json` | `D_chamber_paths.n_true` | 1 |
| [^17] | `outputs/diag_physics_paths_el-90.json` | `cases.전부 켬 (--physics).npaths_median` | 6 |
| [^18] | `outputs/diag_physics_paths_el-90.json` | `cases.회절만 켬.npaths_median` | 14 |